In [1]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.svm import SVR
import xgboost as xgb

import regex as re

from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [2]:
file_path = "..//datasets//final_final_adsorption_done_dataset.csv"
df = pd.read_csv(file_path)
df.shape
df.head()



,adsorbent,source_link,method_processing,surface_area_m2g,particle_size_mm,pore_volume_cm3g,pollutant,initial_concentration_mgL,temperature_c,contact_time_min,qe_mg_g,removal_percent,ph,dose_gL
0,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,17,360,14.93,25,NaN,61.61
1,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,23,360,11.94,20,NaN,61.61
2,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,17,360,6.57,11,NaN,61.61
3,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,20,360,14.33,24,NaN,78.34
4,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,17.5,420,10.72,40,NaN,137.97


In [3]:
## Handle Missing Data

In [4]:
# Create a copy of the dataframe to preserve the original
df_clean = df.copy()

# List of all observed placeholders for missing data
missing_values_list = ['N/P', 'N/A', 'N/A ', 'N/P ', 'N/A,N/A,N/A', 'N/A,N/A']

# Replace all placeholders with numpy.nan
for col in df_clean.columns:
    # Use.replace on the whole dataframe for efficiency
    df_clean.replace(missing_values_list, np.nan, inplace=True)
    # Also handle potential object columns where a value might be just whitespace
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].str.strip()
        df_clean.replace({col: r'^\s*$'}, np.nan, regex=True, inplace=True)

print("Missing values after standardization:")
display(df_clean.isnull().sum())
df_clean.isnull().sum().to_csv('1.csv')

Missing values after standardization:


adsorbent                      0
source_link                    0
method_processing              0
surface_area_m2g              33
particle_size_mm              42
pore_volume_cm3g              56
pollutant                      1
initial_concentration_mgL     16
temperature_c                 13
contact_time_min             125
qe_mg_g                        3
removal_percent              246
ph                            32
dose_gL                      189
dtype: int64

In [5]:
## Handling Ranges, Approximations and Text

In [6]:
# utilizing a function to handle these
def clean_numeric_column(series):
    """
    Cleans a pandas Series intended to be numeric.
    - Removes tilde '~' for approximations.
    - Calculates the mean of ranges (e.g., '10-20').
    - Converts to numeric, coercing errors to NaN.
    """
    # Convert to string to use string methods
    series_str = series.astype(str)
    
    # Remove tildes
    series_str = series_str.str.replace('~', '', regex=False)
    
    # Handle ranges by calculating the mean
    def parse_range(value):
        value = str(value)
        # Handle standard numeric ranges
        if '-' in value and value.replace('-', '', 1).replace('.', '', 2).isdigit():
            try:
                low, high = map(float, value.split('-'))
                return (low + high) / 2.0
            except ValueError:
                return np.nan
        # Handle specific text cases or return the value itself
        else:
            # Remove any non-numeric characters that are not part of a number (e.g., units)
            # This regex keeps numbers, decimals, and the negative sign
            cleaned_value = re.sub(r'[^\d.-]', '', value)
            return cleaned_value

    # Apply the range parsing function
    cleaned_series = series_str.apply(parse_range)
    
    # Convert to numeric, coercing any remaining non-numeric values to NaN
    return pd.to_numeric(cleaned_series, errors='coerce')

# List of columns to apply the numeric cleaning function
numeric_cols_to_clean = [
    'surface_area_m2g', 'particle_size_mm', 'pore_volume_cm3g',
    'initial_concentration_mgL', 'temperature_c', 'contact_time_min',
    'qe_mg_g', 'removal_percent', 'ph', 'dose_gL'
]

# Apply the function to each specified column
for col in numeric_cols_to_clean:
    df_clean[col] = clean_numeric_column(df_clean[col])

# Special handling for 'removal_percent' which had text like 'Increase'
df_clean.replace({'removal_percent': 'Increase'}, np.nan, inplace=True)
df_clean['removal_percent'] = pd.to_numeric(df_clean['removal_percent'], errors='coerce')

print("Data types after numeric cleaning:")
df_clean[numeric_cols_to_clean].info()

Data types after numeric cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   surface_area_m2g           292 non-null    float64
 1   particle_size_mm           283 non-null    float64
 2   pore_volume_cm3g           269 non-null    float64
 3   initial_concentration_mgL  303 non-null    float64
 4   temperature_c              312 non-null    float64
 5   contact_time_min           200 non-null    float64
 6   qe_mg_g                    322 non-null    float64
 7   removal_percent            79 non-null     float64
 8   ph                         293 non-null    float64
 9   dose_gL                    136 non-null    float64
dtypes: float64(10)
memory usage: 25.5 KB


In [7]:
print('Missing values after cleaning and standardizing')
df_clean.isnull().sum()

Missing values after cleaning and standardizing


adsorbent                      0
source_link                    0
method_processing              0
surface_area_m2g              33
particle_size_mm              42
pore_volume_cm3g              56
pollutant                      1
initial_concentration_mgL     22
temperature_c                 13
contact_time_min             125
qe_mg_g                        3
removal_percent              246
ph                            32
dose_gL                      189
dtype: int64

In [8]:
# List of categorical/text columns to standardize
text_cols_to_standardize = ['adsorbent', 'pollutant', 'method_processing']

for col in text_cols_to_standardize:
    # Ensure the column is treated as string type, handling potential NaNs
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip()
    # Replace the string 'nan' that resulted from the astype(str) conversion back to a real NaN
    df_clean.replace({col: 'nan'}, np.nan, inplace=True)

# Example of standardization
print("Sample of standardized 'adsorbent' names:")
print(df_clean['adsorbent'].unique()[:10])

Sample of standardized 'adsorbent' names:
['rh-natural' 'ac-commercial' 'cs-biochar' 'ch-biochar' 'ccac-150'
 'ccac-300' 'ccac-600' 'ccac-bh100' 'ccac-bh200' 'ccac-bh300']


In [9]:
## Feature Engineering with the method_processign column

In [10]:
def engineer_processing_features(df):
    """
    Extracts structured features from the 'method_processing' column.
    """
    df_eng = df.copy()
    
    # Ensure method_processing is a string to avoid errors with NaN
    proc_series = df_eng['method_processing'].fillna('')

    # 1. Pyrolysis/Carbonization Temperature
    # Looks for numbers followed by '°c' or 'c'
    df_eng['pyrolysis_temp_c'] = proc_series.str.extract(r'(\d{3,4})\s?°?c').astype(float)

    # 2. Activation Method and Agent
    # Define keywords for chemical and physical activation
    chemical_agents = {
        'koh': 'KOH', 'naoh': 'NaOH', 'h3po4': '$H_3PO_4$', 'hcl': 'HCl',
        'h2so4': '$H_2SO_4$', 'zncl2': '$ZnCl_2$', 'citric acid': 'Citric Acid'
    }
    
    df_eng['activation_agent'] = 'None'
    for agent_key, agent_name in chemical_agents.items():
        df_eng.loc[proc_series.str.contains(agent_key, na=False), 'activation_agent'] = agent_name
        
    df_eng['is_activated'] = np.where(df_eng['activation_agent']!= 'None', 1, 0)
    
    # 3. Modification Type
    df_eng['is_modified_acid'] = proc_series.str.contains(r'acid|hcl|h2so4|h3po4|hno3', na=False).astype(int)
    df_eng['is_modified_base'] = proc_series.str.contains(r'base|naoh|koh', na=False).astype(int)
    df_eng['is_raw_natural'] = proc_series.str.contains(r'raw|natural|unmodified|untreated', na=False).astype(int)

    return df_eng

df_featured = engineer_processing_features(df_clean)

print("Engineered features from 'method_processing':")
display(df_featured[['method_processing', 'pyrolysis_temp_c', 'activation_agent', 'is_activated', 'is_raw_natural']].sample(10))

Engineered features from 'method_processing':


,method_processing,pyrolysis_temp_c,activation_agent,is_activated,is_raw_natural
320,pyrolysis at 700°c,700.0,None,0,0
101,pyrolysis at 450°c,450.0,None,0,0
300,pyrolysis at 650°c,650.0,None,0,0
62,"raw, washed, dried, ground",NaN,None,0,1
194,pyrolysis at 800°c,800.0,None,0,0
210,pyrolysis at 650°c,650.0,None,0,0
50,physical & chemical (koh) activation,NaN,KOH,1,0
47,physical & chemical (koh) activation,NaN,KOH,1,0
238,pyrolysis at 650°c,650.0,None,0,0
84,pyrolysis at 500°c,500.0,None,0,0


In [11]:
## Hierachical structure for Adsorbent and pollutants

In [12]:
def create_hierarchical_features_refined(df):
    """
    Refines and creates hierarchical categories for adsorbents and pollutants
    using vectorized operations and an expanded keyword list to handle more edge cases.
    """
    df_hier = df.copy()

    # --- Pre-process source columns for robust matching ---
    # Convert to lowercase string and fill NaNs to prevent errors with.str accessor
    adsorbent_series = df_hier['adsorbent'].str.lower().fillna('')
    pollutant_series = df_hier['pollutant'].str.lower().fillna('')
    processing_series = df_hier['method_processing'].str.lower().fillna('')

    # --- Adsorbent Hierarchy (Expanded & Vectorized) ---
    
    # 1. Base Material - Expanded with new keywords from your inspection file
    material_conditions = [
        adsorbent_series.str.contains('rice|rh|oryza sativa|bran'),
        adsorbent_series.str.contains('coconut|cs'),
        adsorbent_series.str.contains('banana'),
        adsorbent_series.str.contains('corn|maize|cc|corncob'),
        adsorbent_series.str.contains('sugarcane|bagasse'),
        adsorbent_series.str.contains('orange|mandarin'),
        adsorbent_series.str.contains('tea'),
        adsorbent_series.str.contains('peanut|groundnut'),
        adsorbent_series.str.contains('wood|pine|sawdust'),
        adsorbent_series.str.contains('straw'),
        adsorbent_series.str.contains('almond'),
        adsorbent_series.str.contains('avocado'),
        adsorbent_series.str.contains('jackfruit'),
        adsorbent_series.str.contains('palm'),
        adsorbent_series.str.contains('sorghum'),
        adsorbent_series.str.contains('bamboo'),
        adsorbent_series.str.contains('tamarind|watermelon|beal|ambarella|peach|apricot|pomegranate|litchi|pomelo|grapefruit|mango|pistachio|cashew|date|olive'),
    ]
    material_choices = [
        'rice_based', 'coconut_based', 'banana_based', 'corn_based', 'sugarcane_based',
        'orange_peel', 'tea_waste', 'peanut_shell', 'wood_based', 'straw_based', 'almond_shell',
        'avocado_waste', 'jackfruit_waste', 'palm_waste', 'sorghum_waste', 'bamboo_based',
        'fruit_waste' # Grouping other rare fruit types
    ]
    df_hier['base_material'] = np.select(material_conditions, material_choices, default='other')

    # 2. Material Class - Using logical order to catch specific types first
    class_conditions = [
        adsorbent_series.str.contains('composite|coated'),
        adsorbent_series.str.contains('hydrochar'),
        adsorbent_series.str.contains('activated carbon|ac|activated charcoal'),
        adsorbent_series.str.contains('biochar|char'),
        df_hier['is_raw_natural'] == 1 # Assumes 'is_raw_natural' column exists
    ]
    class_choices = [
        'composite', 'hydrochar', 'activated_carbon', 'biochar', 'raw_biomass'
    ]
    df_hier['material_class'] = np.select(class_conditions, class_choices, default='unknown_class')

    # --- Pollutant Hierarchy (Expanded & Vectorized) ---
    pollutant_conditions = [
        pollutant_series.str.contains(r'pb|lead|cd|cadmium|cu|copper|zn|zinc|ni|nickel|cr|chromium|hg|mercury|as|arsenic|co|cobalt|mn|manganese|fe|iron'),
        pollutant_series.str.contains('dye|blue|red|violet|green|yellow|orange|ccr'),
        pollutant_series.str.contains('antibiotic|chloroquine|sertraline|carbamazepine|norfloxacin|tetracycline'),
        pollutant_series.str.contains('phenol'),
        pollutant_series.str.contains('nitrogen|nh4|phosphate|nutrient')
    ]
    pollutant_choices = [
        'heavy_metal', 'organic_dye', 'pharmaceutical', 'phenol', 'nutrient'
    ]
    df_hier['pollutant_class'] = np.select(pollutant_conditions, pollutant_choices, default='other_organic')
    
    # --- Feature Extraction from 'method_processing' ---
    # This is better than encoding the whole column. It creates meaningful binary features.
    df_hier['is_acid_treated'] = processing_series.str.contains('acid').astype(int)
    df_hier['is_base_treated'] = processing_series.str.contains('naoh|koh').astype(int)
    df_hier['is_chitosan_modified'] = processing_series.str.contains('chitosan').astype(int)

    return df_hier



# Apply the refined feature engineering function
df_featured_feat = create_hierarchical_features_refined(df_featured)


print("Hierarchical features for adsorbent and pollutant:")
display(df_featured_feat[['adsorbent', 'base_material', 'material_class', 'pollutant', 'pollutant_class']].sample(10))

# Check how many missing values are in the qe_mg_g column
missing_count = df_featured_feat['qe_mg_g'].isna().sum()
print(f"Number of rows with missing qe_mg_g values: {missing_count}")

# Drop rows where qe_mg_g is missing
df = df_featured_feat.dropna(subset=['qe_mg_g'])

# Verify the rows were dropped
print(f"Shape of dataframe after dropping rows: {df.shape}")

# Drop the original high-cardinality and metadata columns
# This is the key to preventing the warning. encode the NEW columns, not the old ones.
columns_to_drop = ['adsorbent', 'pollutant', 'method_processing', 'source_link', 'removal_percent']
df_model_input = df.drop(columns=columns_to_drop)

# check shape after dropping the columns
print(f"Shape of dataframe after dropping rows: {df_model_input.shape}")


# df_model_input.isnull().sum()

Hierarchical features for adsorbent and pollutant:


,adsorbent,base_material,material_class,pollutant,pollutant_class
61,cs-ac,coconut_based,activated_carbon,methylene blue,organic_dye
235,tsac,other,activated_carbon,mb,other_organic
271,tsac,other,activated_carbon,mb,other_organic
11,cs-biochar,coconut_based,biochar,sulfonamides,other_organic
133,ac900,other,activated_carbon,methylene blue (mb),organic_dye
302,sac,other,activated_carbon,methylene blue (mb),organic_dye
285,mc550,other,unknown_class,methylene blue (mb),organic_dye
52,csac,coconut_based,activated_carbon,ammoniacal nitrogen,heavy_metal
153,cmcac,other,activated_carbon,methyl violet (mv),organic_dye
318,bgbhac,other,activated_carbon,methylene blue (mb),organic_dye


Number of rows with missing qe_mg_g values: 3
Shape of dataframe after dropping rows: (322, 26)
Shape of dataframe after dropping rows: (322, 21)


In [13]:
# Separate features and target variables

In [14]:
# Separate Features (X) and the Target Variable (y) ---

X = df_model_input.drop(columns=['qe_mg_g'])

# setting 'qe_mg_g' as primary target
y = df_model_input['qe_mg_g']








In [15]:
# STRATIFIED SPLITTING OF DATASET

In [16]:
# Stratify by Binned Target Variable (Recommended for Regression) ---

print("\n\n--- Stratifying by Binned Target Variable ('qe_mg_g') ---")

# Create bins from the continuous target variable 'y'
# use pd.cut to create 5 bins of equal frequency (quintiles) using qcut.
# This is to ensures each bin has roughly the same number of data points.
y_binned = pd.qcut(y, q=5, labels=False, duplicates='drop')


# Perform the stratified split using these new bins
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y_binned, # Stratify based on the binned target
    random_state=42
)

# --- Verification Step ---
# Check the distribution of the bins
print("Original binned target distribution (%):")
print(round(y_binned.value_counts(normalize=True) * 100, 2))

# We need to create bins for y_train and y_test to verify
y_train_binned = pd.qcut(y_train, q=5, labels=False, duplicates='drop')
y_test_binned = pd.qcut(y_test, q=5, labels=False, duplicates='drop')

print("\nTraining set binned target distribution (%):")
print(round(y_train_binned.value_counts(normalize=True) * 100, 2))

print("\nTest set binned target distribution (%):")
print(round(y_test_binned.value_counts(normalize=True) * 100, 2))



--- Stratifying by Binned Target Variable ('qe_mg_g') ---
Original binned target distribution (%):
qe_mg_g
0    20.19
4    20.19
1    19.88
2    19.88
3    19.88
Name: proportion, dtype: float64

Training set binned target distribution (%):
qe_mg_g
4    20.23
0    20.23
2    19.84
1    19.84
3    19.84
Name: proportion, dtype: float64

Test set binned target distribution (%):
qe_mg_g
0    20.0
4    20.0
2    20.0
3    20.0
1    20.0
Name: proportion, dtype: float64


In [17]:
# Imputation

In [18]:
impute_by_group_cols = ['surface_area_m2g', 'pore_volume_cm3g', 'pyrolysis_temp_c']
# Calculate the median for each 'material_class' from the TRAINING data ONLY.

for col in impute_by_group_cols:
    median_map = X_train.groupby('material_class')[col].median()

    # Use this map to fill missing values in BOTH the training and test sets.
    X_train[col] = X_train[col].fillna(X_train['material_class'].map(median_map))
    X_test[col] = X_test[col].fillna(X_test['material_class'].map(median_map))
    # Handle a potential edge case: If a material_class exists in the test set 
    # but not in the training set, its median will be NaN. We fill these with the global median from the training set.
    glob_median = X_train[col].median()
    X_test.fillna({col: glob_median}, inplace=True)

# After the group-based imputation loop
for col in impute_by_group_cols:
    # Apply global median to any remaining NaNs in both datasets
    glob_median = X_train[col].median()
    X_train.fillna({col: glob_median}, inplace=True)
    X_test.fillna({col: glob_median}, inplace=True)



# --- Global Median Imputation ---
# For columns where a global median is more appropriate
# for temperature, ph
impute_globally_cols = [
    'particle_size_mm', 'initial_concentration_mgL', 'temperature_c',
    'contact_time_min', 'ph', 'dose_gL'
]

for col in impute_globally_cols:
    global_median = X_train[col].median()
    X_train.fillna({col: global_median}, inplace=True)
    X_test.fillna({col: global_median}, inplace=True)


# X_train.isnull().sum()


In [19]:
## Encoding categorical data

In [20]:
# Identify categorical columns for one-hot encoding
categorical_cols = df_model_input.select_dtypes(include=['object', 'category']).columns

# Perform one-hot encoding
# Initialize the encoder
# handle_unknown='ignore' is important. It tells the encoder to ignore (create all-zero columns for)
# any category that appears in the test set but was not in the training set.
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

# Fit on the training data and transform it
X_train_encoded_array = encoder.fit_transform(X_train[categorical_cols])

# Transform the test data using the same fitted encoder
X_test_encoded_array = encoder.transform(X_test[categorical_cols])

# The output is a NumPy array, we need to convert it back to a DataFrame
# and concatenate it with numerical data.

# Create DataFrames with new column names
X_train_encoded = pd.DataFrame(X_train_encoded_array, index=X_train.index, columns=encoder.get_feature_names_out(categorical_cols))
X_test_encoded = pd.DataFrame(X_test_encoded_array, index=X_test.index, columns=encoder.get_feature_names_out(categorical_cols))

# Drop the original categorical columns from X_train and X_test
X_train_final = X_train.drop(columns=categorical_cols)
X_test_final = X_test.drop(columns=categorical_cols)

# Concatenate the numerical data with the new encoded columns
X_train = pd.concat([X_train_final, X_train_encoded], axis=1)
X_test = pd.concat([X_test_final, X_test_encoded], axis=1)

In [21]:
# Handling outliers by capping


In [22]:


print("--- Handling Outliers using IQR Capping ---")

# 1. Identify the numerical columns to cap (same as the ones you scale)
numerical_cols = X_train.select_dtypes(include=np.number).columns
cols_to_cap = [col for col in numerical_cols if X_train[col].nunique() > 2]

# 2. Loop through each column to calculate bounds and apply capping
for col in cols_to_cap:
    # Calculate Q1, Q3, and IQR from the TRAINING data ONLY
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    # Define the upper and lower bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"Capping '{col}': Lower Bound={lower_bound:.2f}, Upper Bound={upper_bound:.2f}")

    # Apply the capping to BOTH the training and test sets
    X_train[col] = np.where(X_train[col] < lower_bound, lower_bound, X_train[col])
    X_train[col] = np.where(X_train[col] > upper_bound, upper_bound, X_train[col])

    X_test[col] = np.where(X_test[col] < lower_bound, lower_bound, X_test[col])
    X_test[col] = np.where(X_test[col] > upper_bound, upper_bound, X_test[col])

print("\nOutlier capping complete.")


--- Handling Outliers using IQR Capping ---
Capping 'surface_area_m2g': Lower Bound=-2740.50, Upper Bound=5159.50
Capping 'particle_size_mm': Lower Bound=-0.42, Upper Bound=1.50
Capping 'pore_volume_cm3g': Lower Bound=-0.34, Upper Bound=1.05
Capping 'initial_concentration_mgL': Lower Bound=-475.00, Upper Bound=925.00
Capping 'temperature_c': Lower Bound=17.50, Upper Bound=37.50
Capping 'contact_time_min': Lower Bound=60.00, Upper Bound=60.00
Capping 'ph': Lower Bound=5.75, Upper Bound=7.75
Capping 'dose_gL': Lower Bound=8.00, Upper Bound=8.00
Capping 'pyrolysis_temp_c': Lower Bound=75.00, Upper Bound=1075.00

Outlier capping complete.


In [23]:
# Interaction features, domain knowledge

In [24]:


print("Original number of features:", X_train.shape[1])

# --- Create new interaction and ratio features ---

# Ratio feature based on domain knowledge
# Add a small constant (1e-6) to the denominator to avoid any potential division-by-zero errors
X_train['conc_dose_ratio'] = X_train['initial_concentration_mgL'] / (X_train['dose_gL'] + 1e-6)
X_test['conc_dose_ratio'] = X_test['initial_concentration_mgL'] / (X_test['dose_gL'] + 1e-6)

# Interaction features based on physical properties
X_train['surface_area_x_pore_volume'] = X_train['surface_area_m2g'] * X_train['pore_volume_cm3g']
X_test['surface_area_x_pore_volume'] = X_test['surface_area_m2g'] * X_test['pore_volume_cm3g']

# Interaction feature based on experimental conditions
X_train['ph_x_temperature'] = X_train['ph'] * X_train['temperature_c']
X_test['ph_x_temperature'] = X_test['ph'] * X_test['temperature_c']

print("New number of features after creating interactions:", X_train.shape[1])



Original number of features: 29
New number of features after creating interactions: 32


In [25]:
## Feature scaling and normalization

In [26]:


# Identify the numerical columns that need scaling.
# This excludes the binary columns created by the OneHotEncoder.
numerical_cols = X_train.select_dtypes(include=np.number).columns
# Filter out the one-hot encoded columns which are typically of type uint8 or int64 and have only 0s and 1s
# A simple way is to find columns that are NOT just 0s and 1s.
cols_to_scale = [col for col in numerical_cols if X_train[col].nunique() > 2]

print("Columns to be scaled:", cols_to_scale)

# Initialize the Scaler
scaler = StandardScaler()

# Create a copy to avoid modifying the original
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit and transform only the selected columns
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("\nFeature scaling complete.")
display(X_train_scaled.head())

Columns to be scaled: ['surface_area_m2g', 'particle_size_mm', 'pore_volume_cm3g', 'initial_concentration_mgL', 'temperature_c', 'ph', 'pyrolysis_temp_c', 'conc_dose_ratio', 'surface_area_x_pore_volume', 'ph_x_temperature']

Feature scaling complete.


,surface_area_m2g,particle_size_mm,pore_volume_cm3g,initial_concentration_mgL,temperature_c,contact_time_min,ph,dose_gL,pyrolysis_temp_c,is_activated,...,material_class_biochar,material_class_hydrochar,material_class_raw_biomass,material_class_unknown_class,pollutant_class_organic_dye,pollutant_class_other_organic,pollutant_class_phenol,conc_dose_ratio,surface_area_x_pore_volume,ph_x_temperature
183,1.178627,-1.131513,-0.252552,1.421481,-0.341146,60.0,0.362382,8.0,1.262319,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.421481,0.153236,-0.214459
323,-0.232640,1.745827,0.168919,-0.607410,-0.341146,60.0,0.980517,8.0,0.672150,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.607410,-0.288445,0.005536
299,-0.833390,1.745827,-0.312762,-0.342936,-0.341146,60.0,-2.213178,8.0,0.377065,0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.342936,-0.788102,-1.131102
92,0.558109,0.062050,1.340307,0.903868,-0.341146,60.0,0.362382,8.0,-0.803273,0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.903868,1.146383,-0.214459
11,-0.956396,-0.236341,-0.252552,-0.418500,-0.341146,60.0,0.362382,8.0,0.672150,1,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.418500,-0.829205,-0.214459


In [27]:
## Modeling

In [28]:
## Hyperparameter Tuning with Cross-Validation

In [29]:
# Hyperparameter Tuning with Cross-Validation for Advanced Models

# Define the Parameter Grid for Random Forest
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 3]
}

# Define the Parameter Grid for XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [4, 3, 5]
}

# Set up and Run GridSearchCV for Random Forest
print("--- 2. Tuning Advanced Model: Random Forest (using 5-fold Cross-Validation) ---")
grid_search_rf = GridSearchCV(estimator=RandomForestRegressor(random_state=42),
                              param_grid=param_grid_rf, cv=5, n_jobs=-1, scoring='r2', verbose=2)
grid_search_rf.fit(X_train_scaled, y_train)
best_rf_model = grid_search_rf.best_estimator_
print(f"Best Random Forest Parameters: {grid_search_rf.best_params_}")

# actual RF model prediction
y_pred_rf = best_rf_model.predict(X_test_scaled)
r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print("Random Forest tuning and evaluation complete.\n")





--- 2. Tuning Advanced Model: Random Forest (using 5-fold Cross-Validation) ---
Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best Random Forest Parameters: {'max_depth': 10, 'min_samples_split': 3, 'n_estimators': 200}
Random Forest tuning and evaluation complete.



In [30]:
## with focus on RF with cross validation

In [31]:
best_rf_model = r2_rf
print(f"The RF cross validated model value is: {r2_rf}")

The RF cross validated model value is: 0.9495582424399298


## Constraint Violation Rate

In [32]:
violations = ((y_pred_rf < 0) | (y_pred_rf > 624)).sum()
violation_rate = (violations / len(y_pred_rf)) * 100
print(f"Actual Violation Rate: {violation_rate:.2f}%")

Actual Violation Rate: 38.46%


## Perturbation Sensitivity

In [33]:
# Example for 1% perturbation
X_test_perturbed = X_test_scaled * 1.01 
# Use the model object (rf_model) to predict on perturbed data, not y_pred_rf
y_pred_perturbed = best_model.predict(X_test_perturbed)  # Replace rf_model with your actual model variable name
# y_pred_rf should already contain predictions from your original data
sensitivity = np.mean(np.abs(y_pred_rf - y_pred_perturbed))
print(f"Actual Perturbation Sensitivity: {sensitivity:.2f} mg/g")

NameError: name 'best_model' is not defined